# BRFSS 2015 — reproducible binary research
No BRFSS results are bundled. Run in Colab (Python 3.12). Complete official codebook review before training. Keep all outputs on Drive; use a new run directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
roots=[Path('/content/drive/MyDrive/BRFSS2015_Research'), Path('/content/drive/MyDrive/Diabetes_Statistics_2015')]
for root in roots:
    print(root, 'exists:',root.exists())
    if root.exists():
        files=[str(p) for p in root.rglob('*') if p.suffix.lower() in ['.csv','.xpt','.parquet']]
        print('Candidate data files:',len(files))
        print('\n'.join(files[:100]))

Copy this entire project folder into Drive first. Set actual paths below after inspecting files. The template deliberately refuses real-data runs until reviewed. Do not reuse a run whose Test has already informed modeling.

In [ ]:
PROJECT=Path('/content/drive/MyDrive/BRFSS2015_Improvement')
DATA=Path('/content/drive/MyDrive/REPLACE/LLCP2015.XPT')
SCHEMA=PROJECT/'schema.reviewed.json'
OUTPUT=PROJECT/'runs'/'run_001'
assert (PROJECT/'src/research.py').is_file()
assert DATA.is_file() and SCHEMA.is_file(), 'Set actual data and reviewed schema paths'

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(PROJECT/'tests/smoke.py')],check=True)

Train-only CV sweep and optimization. Default soft compute budget: 60 hours, 5 folds, 12 Optuna trials for each of two promising boosting families. Leave 12 hours for audit/final/report. Processes do not survive Colab disconnects; use a runtime suitable for this budget. Failures and budget-skipped configurations are logged. The script refuses overwrites.

In [ ]:
subprocess.run([sys.executable,str(PROJECT/'src/research.py'),'train','--data',str(DATA),'--schema',str(SCHEMA),'--output',str(OUTPUT),'--folds','5','--hours','60','--trials','12'],check=True)

In [ ]:
import pandas as pd
display(pd.read_csv(OUTPUT/'04_Model_Training/model_cv_comparison.csv'))
print((OUTPUT/'failures.json').read_text())
print((OUTPUT/'best_model_config.json').read_text())

## Final Test — run once after freezing
Do not tune based on this result. Executing the next cell creates an exclusive final-evaluation lock. A point estimate above 82.26% is not evidence of a comparable improvement over Chang without protocol verification.

In [ ]:
subprocess.run([sys.executable,str(PROJECT/'src/research.py'),'final','--data',str(DATA),'--output',str(OUTPUT)],check=True)
display(pd.read_csv(OUTPUT/'06_Final_Evaluation/final_test_results.csv'))
print((OUTPUT/'06_Final_Evaluation/paired_accuracy_ci.json').read_text())